# Comparing embeddings for the tweet classifier (Colab)

Runs on **Google Colab**, where `gensim` installs without trouble. It rebuilds the same cleaned training data and the same 80/20 validation split as `nlp_tweets.ipynb`, then compares:

- Word2Vec trained on the tweets
- FastText trained on the tweets
- pre-trained GloVe-Twitter-100 (reference), FastText (wiki-news) and Word2Vec (Google News)

Each one is scored alone and combined with TF-IDF. Only `train.csv` is used here; the test set is not touched.

Steps: run the cells in order. In the upload cell, choose your `train.csv`.

In [1]:
!pip install -q gensim

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 42.8 MB/s eta 0:00:00


If a numpy/binary error appears after installing, use **Runtime > Restart session** and continue from the next cell (do not re-run the install cell).

In [2]:
from google.colab import files
uploaded = files.upload()   # choose train.csv

Saving train.csv to train.csv


In [3]:
import re, gc, time
import numpy as np, pandas as pd
from sklearn.model_selection import train_test_split

# ---- same cleaning steps as nlp_tweets.ipynb ----
train = pd.read_csv("train.csv").dropna().drop_duplicates().reset_index(drop=True)
leak = re.compile(r"#(?!ironing\b)\w*(?:sarcas|iron[iy])\w*", flags=re.IGNORECASE)
train["text_noleak"] = train["tweets"].apply(lambda t: leak.sub("", t))
train = train[train["text_noleak"].str.strip().str.len() > 0].reset_index(drop=True)

def clean_tweet(text):
    text = text.lower()
    text = re.sub(r"http\S+|www\.\S+", " ", text)
    text = re.sub(r"@\w+", " ", text)
    text = text.replace("&amp;", " and ")
    text = text.replace("#", " ")
    text = re.sub(r"[^a-z\s']", " ", text)
    return re.sub(r"\s+", " ", text).strip()

train["clean"] = train["text_noleak"].apply(clean_tweet)
train = train[train["clean"] != ""].reset_index(drop=True)
print("Train:", train.shape)   # should print (81342, 4)

y = train["class"]
idx_tr, idx_val = train_test_split(np.arange(len(train)), test_size=0.2, random_state=42, stratify=y)
print("Validation rows:", len(idx_val))   # should print 16269

Train: (81342, 4)
Validation rows: 16269


In [4]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score
from scipy.sparse import hstack, csr_matrix

y_tr, y_val = y.iloc[idx_tr], y.iloc[idx_val]
results = []

def pool_wv(wv, text, oov=False):
    ws = text.split()
    if not oov:
        ws = [w for w in ws if w in wv.key_to_index]
    if not ws:
        return np.zeros(2 * wv.vector_size, dtype=np.float32)
    v = np.vstack([wv[w] for w in ws])
    return np.concatenate([v.mean(axis=0), v.max(axis=0)]).astype(np.float32)

# TF-IDF fitted on the training part only (same settings as before)
tf = TfidfVectorizer(ngram_range=(1, 2), min_df=3, max_features=5000)
Xt_tr = tf.fit_transform(train["clean"].iloc[idx_tr]); Xt_val = tf.transform(train["clean"].iloc[idx_val])
ref = accuracy_score(y_val, LinearSVC(C=0.01, max_iter=5000).fit(Xt_tr, y_tr).predict(Xt_val))
print(f"TF-IDF alone (reference): {ref:.3f}   (earlier result on this split: 0.579)\n")

def evaluate(name, X_emb, extra=""):
    sc = StandardScaler().fit(X_emb[idx_tr])
    Z = sc.transform(X_emb)
    alone = accuracy_score(y_val, LogisticRegression(C=0.1, max_iter=2000).fit(Z[idx_tr], y_tr).predict(Z[idx_val]))
    row = {"embedding": name, "alone": round(alone, 3)}
    for w in [0.05, 0.1]:
        Xa = hstack([Xt_tr, csr_matrix(Z[idx_tr] * w)]).tocsr()
        Xb = hstack([Xt_val, csr_matrix(Z[idx_val] * w)]).tocsr()
        row[f"+TF-IDF (w={w})"] = round(accuracy_score(y_val, LinearSVC(C=0.03, max_iter=5000).fit(Xa, y_tr).predict(Xb)), 3)
    results.append(row)
    print(row, extra)

TF-IDF alone (reference): 0.579   (earlier result on this split: 0.579)



In [5]:
from gensim.models import Word2Vec, FastText
sentences = [t.split() for t in train["clean"]]

t0 = time.time()
w2v = Word2Vec(sentences, vector_size=100, window=5, min_count=2, sg=1, workers=4, epochs=10, seed=42)
print(f"Word2Vec trained in {time.time()-t0:.0f}s, {len(w2v.wv):,} words")
X = np.array([pool_wv(w2v.wv, t) for t in train["clean"]])
evaluate("Word2Vec (own tweets, 100d)", X)

t0 = time.time()
ft = FastText(sentences, vector_size=100, window=5, min_count=2, sg=1, workers=4, epochs=10, seed=42, min_n=3, max_n=6)
print(f"FastText trained in {time.time()-t0:.0f}s, {len(ft.wv):,} words")
X = np.array([pool_wv(ft.wv, t, oov=True) for t in train["clean"]])
evaluate("FastText (own tweets, 100d)", X)

Word2Vec trained in 70s, 32,624 words
{'embedding': 'Word2Vec (own tweets, 100d)', 'alone': 0.569, '+TF-IDF (w=0.05)': 0.586, '+TF-IDF (w=0.1)': 0.584} 
FastText trained in 117s, 32,624 words
{'embedding': 'FastText (own tweets, 100d)', 'alone': 0.555, '+TF-IDF (w=0.05)': 0.58, '+TF-IDF (w=0.1)': 0.58} 


In [7]:
import gensim.downloader as api

# These download once (about 0.4 GB, 1 GB and 1.6 GB) and are freed after each run.
for name, label in [("glove-twitter-100", "GloVe-Twitter-100 (pre-trained)"),
                    ("fasttext-wiki-news-subwords-300", "FastText wiki-news (pre-trained, 300d)"),
                    ("word2vec-google-news-300", "Word2Vec Google News (pre-trained, 300d)")]:
    t0 = time.time()
    kv = api.load(name)
    cover = np.mean([w in kv.key_to_index for w in set(" ".join(train["clean"]).split())])
    print(f"Loaded {name} in {time.time()-t0:.0f}s; covers {cover:.0%} of the tweet vocabulary")
    X = np.array([pool_wv(kv, t) for t in train["clean"]])
    evaluate(label, X)
    del kv, X; gc.collect()

Loaded glove-twitter-100 in 201s; covers 64% of the tweet vocabulary
{'embedding': 'GloVe-Twitter-100 (pre-trained)', 'alone': 0.503, '+TF-IDF (w=0.05)': 0.58, '+TF-IDF (w=0.1)': 0.581} 
[==================================================] 100.0% 958.5/958.4MB downloaded
Loaded fasttext-wiki-news-subwords-300 in 627s; covers 53% of the tweet vocabulary
{'embedding': 'FastText wiki-news (pre-trained, 300d)', 'alone': 0.531, '+TF-IDF (w=0.05)': 0.581, '+TF-IDF (w=0.1)': 0.58} 
[==================================================] 100.0% 1662.8/1662.8MB downloaded
Loaded word2vec-google-news-300 in 269s; covers 49% of the tweet vocabulary
{'embedding': 'Word2Vec Google News (pre-trained, 300d)', 'alone': 0.533, '+TF-IDF (w=0.05)': 0.581, '+TF-IDF (w=0.1)': 0.58} 


In [8]:
table = pd.DataFrame(results)
table.loc[len(table)] = ["TF-IDF alone (reference)", None, ref, ref]
print(table.to_string(index=False))

                               embedding  alone  +TF-IDF (w=0.05)  +TF-IDF (w=0.1)
             Word2Vec (own tweets, 100d)  0.569          0.586000         0.584000
             FastText (own tweets, 100d)  0.555          0.580000         0.580000
         GloVe-Twitter-100 (pre-trained)  0.503          0.580000         0.581000
  FastText wiki-news (pre-trained, 300d)  0.531          0.581000         0.580000
Word2Vec Google News (pre-trained, 300d)  0.533          0.581000         0.580000
                TF-IDF alone (reference)    NaN          0.578585         0.578585


/tmp/ipykernel_4941/2184411071.py:2: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  table.loc[len(table)] = ["TF-IDF alone (reference)", None, ref, ref]
